<a href="https://colab.research.google.com/github/joannadulay/FlyRank-ML-Internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [ ]:
!pip install -q datasets duckdb scikit-learn huggingface_hub

from google.colab import userdata
from huggingface_hub import login
from datasets import load_dataset
import duckdb
import pandas as pd
import numpy as np

login(token=userdata.get("HF_TOKEN"))

MONTH = "2026-03"  # same mid-panel dev month as the ML-04 data contract

ds = load_dataset(
    "FlyRank/internship-warehouse",
    data_files={"train": f"fact_content_daily_performance/month={MONTH}/data_0.parquet"},
    split="train",
)
df = ds.to_pandas()

con = duckdb.connect()
con.register("panel", df)

# --- Contract carried over from ML-04 ---
# GSC availability still gates GSC features: a 0 in gsc_impressions on an
# unavailable row means "nothing recorded," not "zero impressions." That
# distinction can\'t be safely filled, so rows without GSC data are dropped
# rather than filled.
feat = con.execute("""
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,
        ga4_data_available,
        ga4_engaged_sessions,
        sessions_organic
    FROM panel
    WHERE gsc_data_available IS TRUE
""").df()

print("Rows after GSC-availability filter:", feat.shape[0])

# --- Fill: GA4 fields, not GSC fields ---
# ML-04 showed only 3.7% of March rows have BOTH sources. Requiring GA4
# availability here (like ML-04 did) would throw away ~96% of the usable
# GSC signal, so instead of dropping: when ga4_data_available is not True,
# fill ga4_engaged_sessions and sessions_organic with 0.
#
# CAVEAT (explicit, not hidden): this treats "GA4 wasn\'t tracking this
# page-day" the same as "GA4 tracked it and saw zero engagement." Those are
# NOT the same thing -- a page with no GA4 tag installed reads identically
# to a page that genuinely got zero engaged sessions. This is a real
# approximation, kept only because the alternative (dropping to the 3.7%
# overlap) loses far more rows than it fixes. Named again in Section 4.
# BUG FOUND ON FIRST RUN: `feat["ga4_data_available"] != True` looked right
# but silently failed on the truly-missing rows. duckdb/pandas represents
# "no info" as NA, not False -- and `NA != True` evaluates to NA, not True.
# A mask containing NA gets treated by pandas as "don\'t select" rather than
# raising, so exactly the rows this fill was meant to catch (client
# client_73cda7b4e4f265ea, which has no GA4 at all) fell straight through
# and stayed NaN. Only the explicit False rows got filled. Fix: coerce to a
# real boolean first, so "False" and "missing" collapse into one bucket
# instead of "missing" silently meaning "skip."
ga4_available = feat["ga4_data_available"].fillna(False).astype(bool)
ga4_missing = ~ga4_available
feat.loc[ga4_missing, ["ga4_engaged_sessions", "sessions_organic"]] = 0

# Verify the fix actually worked -- a contract claim without a check next
# to it is a guess (same rule as ML-04 Section 3).
remaining_nulls = feat[["ga4_engaged_sessions", "sessions_organic"]].isna().sum()
print("Remaining NaNs after fill (should be 0, 0):")
print(remaining_nulls)

# --- Engineered feature: click-through rate ---
# CTR = clicks / impressions. Known at report_date, same-day, no forward
# looking -- built from the same two columns already in the honest feature
# set, so it adds no new leakage surface. Guard divide-by-zero: 0
# impressions -> CTR defined as 0, not NaN. That 0 IS a real zero (no
# impressions means no opportunity to click) -- unlike the GA4 case above,
# there\'s no ambiguity here since gsc_data_available IS TRUE already holds.
feat["gsc_ctr"] = np.where(
    feat["gsc_impressions"] > 0,
    feat["gsc_clicks"] / feat["gsc_impressions"],
    0.0,
)

# --- Flag, don\'t guess: gsc_avg_position == 0 ---
# Verified in Section 2\'s check: 163,189 rows show gsc_avg_position ==
# 0.0 exactly, and roughly half of those have gsc_impressions > 1 -- too
# common to dismiss as a low-sample rounding artifact, but not confirmed
# as a sentinel value either. Since 0 is the best possible rank (positions
# are 1-indexed), a wrong guess here is not neutral noise -- it reads as
# "amazing performance" in the worst case. Rather than silently trusting
# it or silently dropping/filling it (either one is an unverified guess in
# a different direction), flag it and let downstream steps decide.
feat["gsc_position_suspect"] = feat["gsc_avg_position"] == 0

feat = feat.drop(columns=["ga4_data_available"])

print("Feature frame shape:", feat.shape)
print("Rows with a GA4 fill applied:", int(ga4_missing.sum()))
print("Rows flagged gsc_position_suspect:", int(feat["gsc_position_suspect"].sum()))
feat.head()

# No categorical fields in this pass -- the contract\'s five base columns
# are all numeric, and gsc_ctr is a numeric ratio of two of them. Deferred,
# not silently skipped: see Section 4 for why.

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows after GSC-availability filter: 3611061
Remaining NaNs after fill (should be 0, 0):
ga4_engaged_sessions    0
sessions_organic        0
dtype: int64
Feature frame shape: (3611061, 10)
Rows with a GA4 fill applied: 3246714
Rows flagged gsc_position_suspect: 163189


,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_engaged_sessions,sessions_organic,gsc_ctr,gsc_position_suspect
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,2026-03-01,20,0,3.350000,0.0,0.0,0.000,False
1,client_73cda7b4e4f265ea,content_05597932fe4da067,2026-03-01,1,0,0.000000,0.0,0.0,0.000,True
2,client_73cda7b4e4f265ea,content_7a105f548d9c6916,2026-03-01,125,1,4.928000,0.0,0.0,0.008,False
3,client_73cda7b4e4f265ea,content_905aa32a0230694e,2026-03-01,7,0,4.000000,0.0,0.0,0.000,False
4,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,2026-03-01,11,0,2.272727,0.0,0.0,0.000,False


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [ ]:
# gsc_impressions
#   Meaning: count of times the page appeared in Google Search results on
#            report_date.
#   Missing: never missing within this feature frame -- rows where GSC
#            wasn't tracked are dropped upstream (Section 1), not filled.
#   Categorical: no, a same-day count.
#   Available-when: known as of report_date; GSC logs it for the day that
#            already happened, no forward-looking aggregation.
#
# gsc_clicks
#   Meaning: count of clicks from search results to the page on report_date.
#   Missing: same as gsc_impressions -- dropped, not filled, upstream.
#   Categorical: no.
#   Available-when: same-day GSC metric, no look-ahead.
#
# gsc_avg_position
#   Meaning: average search-results ranking position across that day's
#            impressions.
#   Missing: dropped upstream with the rest of GSC -- confirmed 0 actual
#            NaNs in this column (verified below). BUT: 163,189 rows
#            (out of 3.61M) show gsc_avg_position == 0.0 exactly, and only
#            about half of those (84,438) are low-impression rows
#            (impressions <= 1). The other 78,751 have impressions > 1,
#            some with real clicks -- so this isn't just a low-sample
#            rounding quirk. A genuine average GSC position of exactly 0
#            is unusual (positions are 1-indexed), so this looks more like
#            a placeholder for "position not tracked" than a real average
#            -- a hypothesis, not confirmed here. Rather than guess in
#            either direction, Section 1 now carries a companion column
#            gsc_position_suspect (True when gsc_avg_position == 0) so the
#            ambiguity is visible and queryable instead of silently
#            resolved one way or the other.
#   Categorical: no.
#   Available-when: same-day.
#
# ga4_engaged_sessions
#   Meaning: count of GA4-defined "engaged" sessions (meaningful
#            interaction, not just a pageview) on report_date.
#   Missing: filled with 0 when ga4_data_available is not True (Section 1).
#            CAVEAT carried forward: this makes "no GA4 tag on this page"
#            look identical to "GA4 tracked it and saw zero engagement."
#            ~90% of the filled feature frame is this manufactured 0, not
#            an observation (see Section 4).
#   Categorical: no.
#   Available-when: same-day GA4 metric in principle. NOT YET VERIFIED:
#            GA4 can have processing lag of hours, so "available at
#            report_date" may not literally hold at the instant that date
#            ends -- a scheduling detail, not something confirmed here.
#
# sessions_organic
#   Meaning: count of sessions on report_date attributed to organic search
#            as the traffic source, from GA4.
#   Missing: same fill treatment and same caveat as ga4_engaged_sessions.
#   Categorical: no.
#   Available-when: same as ga4_engaged_sessions, same unverified lag note.
#
# gsc_ctr (engineered)
#   Meaning: clicks / impressions for that day -- a ratio of two columns
#            already in this table.
#   Missing: not naturally missing -- both inputs are guaranteed present
#            once gsc_data_available IS TRUE. Zero-impression rows are
#            defined as CTR = 0 by convention (Section 1), not left NaN.
#   Categorical: no.
#   Available-when: derived entirely from same-day GSC columns, so it
#            inherits their availability -- no new forward-looking risk.
#
# Context columns (client_hash_id, content_hash_id, report_date)
#   Not features -- they identify the row per the ML-04 contract, carried
#   over unchanged. Assumed non-null and always available; not verified
#   again here since ML-04 Query 1 already proved the grain.
#
# gsc_position_suspect (engineered flag, added in Section 1)
#   Meaning: True when gsc_avg_position == 0.0 -- a boolean marker for the
#            "genuine average of exactly 0" ambiguity found while checking
#            the claim above (163,189 of 3.61M rows, roughly half of them
#            at gsc_impressions > 1, so not purely a low-sample artifact).
#   Missing: never missing -- it's computed from gsc_avg_position, which
#            is itself never null in this feature frame.
#   Categorical: YES -- this is the one categorical (binary) field in the
#            set, which corrects the earlier draft of this section that
#            claimed there were none. It exists precisely because a
#            numeric column (gsc_avg_position) couldn't be trusted at
#            face value; the flag is the categorical handling for that
#            ambiguity, not a separate design choice.
#   Available-when: same-day, inherits gsc_avg_position's availability --
#            no new forward-looking risk, since it's a same-day
#            recomputation, not a new data source.
#   NOT YET VERIFIED: whether this flag leaks against a clicks-derived
#            label -- it's not built from gsc_clicks, but "not derived
#            from the label column" isn't the same as "proven clean."
#            Tested directly in Section 3.

In [ ]:
# Verify claim flagged in Section 2: is gsc_avg_position ever a
# null/zero artifact on very-low-impression rows, rather than a real
# average? A contract claim without a query next to it is a guess.

nulls_in_position = feat["gsc_avg_position"].isna().sum()
print("NaNs in gsc_avg_position:", nulls_in_position)

zero_position = (feat["gsc_avg_position"] == 0).sum()
print("Rows where gsc_avg_position == 0:", zero_position)

low_impr = feat["gsc_impressions"] <= 1
print("Total rows with gsc_impressions <= 1:", low_impr.sum())
print("Of the zero-position rows, how many also have gsc_impressions <= 1:",
      ((feat["gsc_avg_position"] == 0) & low_impr).sum())
print("Of the zero-position rows, how many have gsc_impressions > 1 (i.e. NOT a low-impression artifact):",
      ((feat["gsc_avg_position"] == 0) & ~low_impr).sum())

feat[feat["gsc_avg_position"] == 0][["gsc_impressions", "gsc_clicks", "gsc_avg_position"]].head(10)

NaNs in gsc_avg_position: 0
Rows where gsc_avg_position == 0: 163189
Total rows with gsc_impressions <= 1: 386362
Of the zero-position rows, how many also have gsc_impressions <= 1: 84438
Of the zero-position rows, how many have gsc_impressions > 1 (i.e. NOT a low-impression artifact): 78751


,gsc_impressions,gsc_clicks,gsc_avg_position
1,1,0,0.0
15,1,0,0.0
23,2,0,0.0
24,1,0,0.0
27,2,0,0.0
30,1,0,0.0
34,1,0,0.0
51,3,0,0.0
52,1,0,0.0
87,1,0,0.0


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score

# --- Toy proxy label (March-only, same mechanic as ML-04) ---
# Not the real Growth/Recovery/Stable/Decline label -- that needs a window
# past March. This is a stand-in built purely to have something to leak
# into.
median_clicks = feat["gsc_clicks"].median()
feat["toy_label"] = (feat["gsc_clicks"] > median_clicks).astype(int)

# Baseline: features with no relationship to gsc_clicks at all.
honest_features = [
    "gsc_impressions", "gsc_avg_position",
    "ga4_engaged_sessions", "sessions_organic",
]

def run(extra_features):
    cols = honest_features + extra_features
    X, y = feat[cols], feat["toy_label"]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=42, stratify=y
    )
    clf = DecisionTreeClassifier(max_depth=4, random_state=42)
    clf.fit(X_train, y_train)
    return f1_score(y_test, clf.predict(X_test), average="macro")

honest_f1 = run([])
print(f"Honest macro F1 (baseline, no click-derived columns): {honest_f1:.3f}")

# --- Attack 1: gsc_clicks itself ---
# Same trap as ML-04 -- this IS the column the label is thresholded from.
clicks_f1 = run(["gsc_clicks"])
print(f"+ gsc_clicks:            {clicks_f1:.3f}  (jump: +{clicks_f1 - honest_f1:.3f})")

# --- Attack 2: gsc_ctr -- this notebook's NEW engineered feature ---
# gsc_ctr = gsc_clicks / gsc_impressions. It doesn't say "clicks" in the
# name and it wasn't in ML-04's feature set, but it's built directly from
# the same column the label is thresholded on. The lesson: an engineered
# feature inherits leakage from its inputs -- renaming or transforming a
# leaky column does not make it clean.
ctr_f1 = run(["gsc_ctr"])
print(f"+ gsc_ctr:               {ctr_f1:.3f}  (jump: +{ctr_f1 - honest_f1:.3f})")

# --- Attack 3: gsc_position_suspect -- the flag added in Section 1 ---
# NOT derived from gsc_clicks at all -- it comes from gsc_avg_position.
# Testing it anyway, since "not derived from the label column" is not the
# same as "proven clean." If this jumps too, the flag is correlated with
# clicks through some other path and needs the same scrutiny as the two
# attacks above.
flag_f1 = run(["gsc_position_suspect"])
print(f"+ gsc_position_suspect:  {flag_f1:.3f}  (jump: +{flag_f1 - honest_f1:.3f})")

# RESULTS (one train/test split, fixed random_state=42, against the
# real 3.61M-row feature frame -- one run, not cross-validated):
#   Honest baseline:        0.785
#   + gsc_clicks:           1.000  (jump: +0.215)
#   + gsc_ctr:              1.000  (jump: +0.215) -- SAME jump as raw clicks
#   + gsc_position_suspect: 0.785  (jump: +0.000) -- no leak at all
#
# VERDICT:
#   gsc_clicks and gsc_ctr both leaked in this run, by an identical
#   margin. Engineering the ratio did not launder the leak -- it's just
#   as diagnostic of a clicks-based label as the raw column it's built
#   from.
#   Fix: keep BOTH out of any model trained against a clicks-derived
#   label (real or toy).
#
#   gsc_position_suspect showed no jump. It lives in the same feature
#   frame as two leaky columns but was not itself correlated with clicks
#   in this test. That's a measured result, not an assumption -- it
#   earned the "clean" label by testing, the same way gsc_clicks and
#   gsc_ctr earned "leaky."

Honest macro F1 (baseline, no click-derived columns): 0.785
+ gsc_clicks:            1.000  (jump: +0.215)
+ gsc_ctr:               1.000  (jump: +0.215)
+ gsc_position_suspect:  0.785  (jump: +0.000)


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [ ]:
# 1. Rows where gsc_data_available is False
#    Carried over from the ML-04 contract: a 0 there means "not recorded,"
#    not "zero impressions." Excluding, not filling -- filling would fake
#    an observation that never happened.
#
# 2. gsc_clicks, as a model INPUT (against a clicks-derived label)
#    Observed leaky in Section 3 -- macro F1 jumped to 1.000 in that run.
#    Kept in the table only as the input to gsc_ctr's arithmetic, never
#    exposed to a classifier directly.
#
# 3. gsc_ctr, as a model INPUT (against a clicks-derived label)
#    Observed leaky in Section 3, same +0.215 jump as raw gsc_clicks --
#    turning a leaky column into a ratio did not launder the leak.
#
# 4. client_hash_id / content_hash_id, as direct model features
#    PRIVACY: both are pre-hashed, not raw client names or URLs, so they
#    satisfy the "no client names/URLs" rule on their own. Excluded from
#    modeling anyway, on structural (not tested) grounds: a high-cardinality
#    ID let into a tree lets it memorize per-client/per-page baselines
#    instead of generalizing from the six real signals -- a classic
#    overfitting/leakage vector even without a clicks-style proof.
#
# 5. report_date, as a raw model feature
#    Pure context per the ML-04 contract -- identifies WHEN a row happened,
#    not a predictive signal by itself. A derived day-of-week/day-of-month
#    feature might be worth testing in a future pass; not attempted here.
#
# 6. sessions_direct, sessions_referral, sessions_social, ga4_pageviews,
#    ga4_sessions, ga4_users, total_engagement_sec
#    Same-day and technically available (noted in ML-04), but deliberately
#    not engineered or tested this round -- deferred, not evaluated. This
#    is a different status than #2/#3: those were tested and found leaky,
#    these simply weren't looked at yet.
#
# 7. client_has_gsc / client_has_ga4
#    Client-level setup/config flags, not that page's performance on that
#    day. Used to build the contract's reasoning, not fed to a model.
#
# PRIVACY CHECK: no client names, URLs, or private query text appear
# anywhere in this notebook or the feature frame -- every identifying
# column is a pre-hashed ID inherited from the source table, consistent
# with the self-check requirement below.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.